# Baseline Multilayer Perceptron (MLP)

In [13]:
#| output: false

import sys
from pathlib import Path

# Set to root for src load
root = Path.cwd()
while root != root.parent and not (root / "src").exists():
    root = root.parent

if not (root / "src").exists():
    raise FileNotFoundError("Could not find 'src' folder above current working directory.")

sys.path.insert(0, str(root))
print("Project root:", root)

Project root: c:\Users\joetn\CS273P_MachineLearning_Final_Project


In [26]:
import json
import libpysal
import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
from matplotlib.colors import ListedColormap
import seaborn as sns
from IPython.display import display

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.feature_selection import mutual_info_classif
from sklearn.cluster import KMeans
import torch
from torch.utils.data import TensorDataset, DataLoader

# Set plt plotting configs
plt.rcParams.update({
    "font.size": 9,
    "axes.titlesize": 10,
    "axes.labelsize": 9,
    "xtick.labelsize": 8,
    "ytick.labelsize": 8,
    "legend.fontsize": 8,
    "legend.title_fontsize": 9,
    "legend.frameon": True,
    "legend.borderpad": 0.3
})

SEED = 42

In [ ]:
# Load artifacts
geo_df = gpd.read_file("../data/processed/svi_counties.gpkg", layer="counties")

X = pd.read_csv("../data/processed/X.csv")

X_log = pd.read_csv("../data/processed/X_log.csv")

y = pd.read_csv("../data/processed/y.csv")

with open("../data/processed/feature_titles.json") as f:
    feature_titles_log = json.load(f)

## Model Motivation

The multilayer perceptron (MLP) serves as the tabular deep learning baseline for this study. Its purpose is to establish a nonlinear benchmark for predicting county-level Social Vulnerability Index (SVI) theme rankings using only the transformed EP_* indicator features, without incorporating any explicit spatial structure. This makes the MLP a useful reference point for evaluating whether later graph-based models improve performance by modeling geographic relationships between counties.

An MLP is appropriate in this setting because the EP_* indicators represent multiple interrelated socioeconomic, demographic, and housing characteristics whose relationships with SVI outcomes may be nonlinear. Compared with linear regression models, the MLP can learn more flexible mappings between the feature matrix and the target variables while remaining conceptually simple and computationally efficient. At the same time, it still treats counties as independent observations, which provides a meaningful contrast with the graph neural network models developed later.

## Model Formulation

Let $X \in \mathbb{R}^{n \times p}$ denote the feature matrix, where $n$ is the number of counties and $p = 16$ is the number of transformed EP_* predictor variables. Let $Y \in \mathbb{R}^{n \times q}$ denote the target matrix, where $q = 4$ corresponds to the four SVI theme rankings: `RPL_THEME1`, `RPL_THEME2`, `RPL_THEME3`, and `RPL_THEME4`.

The MLP defines a nonlinear mapping

$$
f_{\theta}: \mathbb{R}^{p} \rightarrow \mathbb{R}^{q},
$$

parameterized by weights and biases $\theta$, such that the predicted SVI themes are

$$
\hat{Y} = f_{\theta}(X).
$$

For the baseline model, the network consists of fully connected layers with nonlinear activation functions applied between hidden layers. For an input vector $x_i \in \mathbb{R}^{p}$, the forward pass can be written as

$$
h^{(1)}_i = \phi(W^{(1)} x_i + b^{(1)}),
$$

$$
h^{(2)}_i = \phi(W^{(2)} h^{(1)}_i + b^{(2)}),
$$

$$
\hat{y}_i = W^{(3)} h^{(2)}_i + b^{(3)},
$$

where $\phi(\cdot)$ is the activation function, $h^{(1)}_i$ and $h^{(2)}_i$ are hidden representations, and $\hat{y}_i \in \mathbb{R}^{4}$ is the predicted vector of SVI theme values for county $(i)$. Because the task is **multi-output regression**, the final layer is linear and produces continuous outputs.

Model parameters are estimated by minimizing a regression loss over the training data. Using mean squared error (MSE), the objective is

$$
\mathcal{L}(\theta) = \frac{1}{n} \sum_{i=1}^{n} \lVert y_i - \hat{y}_i \rVert_2^2.
$$

This formulation allows the MLP to learn shared nonlinear structure across the four SVI themes while predicting all targets jointly within a single model.

## Baseline Architecture

The baseline MLP uses a compact two-hidden-layer design:

$$
16 \rightarrow 64 \rightarrow 32 \rightarrow 4
$$

with ReLU activations and light dropout regularization between hidden layers. 

This baseline therefore acts as a strong tabular benchmark while preserving a clear methodological distinction between feature-only learning and later spatial graph-based learning.

## Ablation Plan

After establishing the baseline MLP, a series of controlled ablation experiments will be used to evaluate how sensitive performance is to architectural and preprocessing choices. These ablations are designed to clarify whether any gains come from meaningful modeling improvements rather than arbitrary hyperparameter choices.

The main ablation directions are:

1. Feature Representation

    The first ablation compares **raw EP_* features** against the **log-transformed feature set** motivated by the EDA. This tests whether reducing skewness and heavy-tailed behavior improves optimization and predictive performance.

2. Network Capacity

    The second ablation examines architectural complexity by comparing a **smaller shallow network**, the **baseline two-layer model**, and a **wider or deeper variant**. This helps determine whether additional nonlinear capacity meaningfully improves county-level SVI prediction.

3. Regularization

    The third ablation studies the effect of **dropout strength** and, if needed, **weight decay**. This evaluates whether the baseline is under-regularized or over-regularized and helps justify the final model configuration.

Together, these ablations provide a structured way to assess the robustness of the MLP baseline before moving to graph-based models such as the GNN and GCN.


In [ ]:
# First split: train vs temp (val + test) = 70/30
X_train, X_temp, Xlog_train, Xlog_temp, y_train, y_temp = train_test_split(
    X,
    X_log,
    y,
    test_size=0.30,
    random_state=SEED
)

# Second split: validation vs test = 15/15
X_val, X_test, Xlog_val, Xlog_test, y_val, y_test = train_test_split(
    X_temp,
    Xlog_temp,
    y_temp,
    test_size=0.50,
    random_state=SEED
)

In [ ]:
# Scale X splits
scaler_X = StandardScaler()

X_train_scaled = scaler_X.fit_transform(X_train)
X_val_scaled = scaler_X.transform(X_val)
X_test_scaled = scaler_X.transform(X_test)

# Scale X_log splits
scaler_Xlog = StandardScaler()

Xlog_train_scaled = scaler_Xlog.fit_transform(Xlog_train)
Xlog_val_scaled = scaler_Xlog.transform(Xlog_val)
Xlog_test_scaled = scaler_Xlog.transform(Xlog_test)

# Scale y splits
scaler_y = StandardScaler()

y_train_scaled = scaler_y.fit_transform(y_train)
y_val_scaled = scaler_y.transform(y_val)
y_test_scaled = scaler_y.transform(y_test)

In [ ]:
#| output: false

# Data Validation
split_summary = pd.DataFrame({
    "Dataset": ["Train", "Validation", "Test"],
    "Samples": [
        X_train.shape[0],
        X_val.shape[0],
        X_test.shape[0]
    ],
    "Feature Count (X)": [
        X_train.shape[1],
        X_val.shape[1],
        X_test.shape[1]
    ],
    "Target Count (y)": [
        y_train.shape[1],
        y_val.shape[1],
        y_test.shape[1]
    ]
})

display(split_summary.style.hide(axis="index"))

Dataset,Samples,Feature Count (X),Target Count (y)
Train,2200,16,4
Validation,472,16,4
Test,472,16,4


In [ ]:
#| output: false

# Data Validation
scale_check = pd.DataFrame({
    "Variable Group": ["Raw Features (X)", "Log-Transformed Features (X_log)", "Targets (y)"],
    "Training Mean (≈0)": [
        round(X_train_scaled.mean(), 4),
        round(Xlog_train_scaled.mean(), 4),
        round(y_train_scaled.mean(), 4)
    ],
    "Training Std (≈1)": [
        round(X_train_scaled.std(), 4),
        round(Xlog_train_scaled.std(), 4),
        round(y_train_scaled.std(), 4)
    ]
})

display(scale_check.style.hide(axis="index"))

Variable Group,Training Mean (≈0),Training Std (≈1)
Raw Features (X),0.000000,1.000000
Log-Transformed Features (X_log),-0.000000,1.000000
Targets (y),-0.000000,1.000000


In [ ]:
#| output: false

# Raw feature tensors
Xraw_train_t = torch.tensor(X_train_scaled, dtype=torch.float32)
Xraw_val_t   = torch.tensor(X_val_scaled, dtype=torch.float32)
Xraw_test_t  = torch.tensor(X_test_scaled, dtype=torch.float32)

# Log-transformed feature tensors
Xlog_train_t = torch.tensor(Xlog_train_scaled, dtype=torch.float32)
Xlog_val_t   = torch.tensor(Xlog_val_scaled, dtype=torch.float32)
Xlog_test_t  = torch.tensor(Xlog_test_scaled, dtype=torch.float32)

# Target tensors
y_train_t = torch.tensor(y_train_scaled, dtype=torch.float32)
y_val_t   = torch.tensor(y_val_scaled, dtype=torch.float32)
y_test_t  = torch.tensor(y_test_scaled, dtype=torch.float32)

# Data Validation
tensor_shapes = pd.DataFrame({
    "Dataset": ["Train", "Validation", "Test"],
    "Raw Feature Tensor Shape": [
        tuple(Xraw_train_t.shape),
        tuple(Xraw_val_t.shape),
        tuple(Xraw_test_t.shape)
    ],
    "Log Feature Tensor Shape": [
        tuple(Xlog_train_t.shape),
        tuple(Xlog_val_t.shape),
        tuple(Xlog_test_t.shape)
    ],
    "Target Tensor Shape": [
        tuple(y_train_t.shape),
        tuple(y_val_t.shape),
        tuple(y_test_t.shape)
    ]
})

display(tensor_shapes.style.hide(axis="index"))

Dataset,Raw Feature Tensor Shape,Log Feature Tensor Shape,Target Tensor Shape
Train,"(2200, 16)","(2200, 16)","(2200, 4)"
Validation,"(472, 16)","(472, 16)","(472, 4)"
Test,"(472, 16)","(472, 16)","(472, 4)"


In [30]:
#| label: tbl-mlp-dataloader-summary
#| tbl-cap: "Summary of TensorDataset and DataLoader configurations for the baseline MLP experiments using both standardized raw EP_* features and standardized log-transformed EP_* features. The log-transformed feature pipeline is used for the baseline model, while the raw feature pipeline is retained for later ablation analysis."

# Set batch size
BATCH_SIZE = 64

# Raw feature datasets/loaders
train_ds_raw = TensorDataset(Xraw_train_t, y_train_t)
val_ds_raw   = TensorDataset(Xraw_val_t, y_val_t)
test_ds_raw  = TensorDataset(Xraw_test_t, y_test_t)

train_loader_raw = DataLoader(train_ds_raw, batch_size=BATCH_SIZE, shuffle=True)
val_loader_raw   = DataLoader(val_ds_raw, batch_size=BATCH_SIZE, shuffle=False)
test_loader_raw  = DataLoader(test_ds_raw, batch_size=BATCH_SIZE, shuffle=False)

# Log feature datasets/loaders
train_ds_log = TensorDataset(Xlog_train_t, y_train_t)
val_ds_log   = TensorDataset(Xlog_val_t, y_val_t)
test_ds_log  = TensorDataset(Xlog_test_t, y_test_t)

train_loader_log = DataLoader(train_ds_log, batch_size=BATCH_SIZE, shuffle=True)
val_loader_log   = DataLoader(val_ds_log, batch_size=BATCH_SIZE, shuffle=False)
test_loader_log  = DataLoader(test_ds_log, batch_size=BATCH_SIZE, shuffle=False)

dataloader_summary = pd.DataFrame({
    "Pipeline": [
        "Raw Features",
        "Raw Features",
        "Raw Features",
        "Log-Transformed Features",
        "Log-Transformed Features",
        "Log-Transformed Features"
    ],
    "Dataset": [
        "Train", "Validation", "Test",
        "Train", "Validation", "Test"
    ],
    "Samples": [
        len(train_ds_raw), len(val_ds_raw), len(test_ds_raw),
        len(train_ds_log), len(val_ds_log), len(test_ds_log)
    ],
    "Batch Size": [BATCH_SIZE] * 6,
    "Num Batches": [
        len(train_loader_raw), len(val_loader_raw), len(test_loader_raw),
        len(train_loader_log), len(val_loader_log), len(test_loader_log)
    ],
})

display(dataloader_summary.style.hide(axis="index"))

Pipeline,Dataset,Samples,Batch Size,Num Batches
Raw Features,Train,2200,64,35
Raw Features,Validation,472,64,8
Raw Features,Test,472,64,8
Log-Transformed Features,Train,2200,64,35
Log-Transformed Features,Validation,472,64,8
Log-Transformed Features,Test,472,64,8
